# pandas: The Complete Practical Guide

Notebook for **Chapter 17** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch17.html

Tip: change a number, predict the output, then run the cell again.

## 17.1 What pandas Is, When to Use It, and the Mental Model

In [ ]:
import pandas as pd
import numpy as np

print(pd.__version__)
print(np.__version__)
print(pd.options.display.max_rows)

3.0.6
2.5.3
60


## 17.2 Series, DataFrame and Index

In [ ]:
import pandas as pd

s = pd.Series([10, 7, 3], index=["Oslo", "Rome", "Lima"], name="sales")
print(s)
print(s["Rome"], s.iloc[0])
print(s.index)
print(s.dtype)

Oslo    10
Rome     7
Lima     3
Name: sales, dtype: int64
7 10
Index(['Oslo', 'Rome', 'Lima'], dtype='str')
int64


In [ ]:
import pandas as pd

rows = [
    {"city": "Oslo", "sales": 10, "paid": True},
    {"city": "Rome", "sales": 7, "paid": False},
    {"city": "Oslo", "sales": 3, "paid": True},
]
df = pd.DataFrame(rows)
print(df)
print(df.dtypes)

a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["y", "z", "w"])
print(a + b)

   city  sales   paid
0  Oslo     10   True
1  Rome      7  False
2  Oslo      3   True
city       str
sales    int64
paid      bool
dtype: object
w     NaN
x     NaN
y    12.0
z    23.0
dtype: float64


In [ ]:
import pandas as pd

df = pd.DataFrame({"id": [101, 102, 103], "name": ["Ana", "Bo", "Cy"], "score": [88, 92, 75]})
by_id = df.set_index("id")
print(by_id)
print(by_id.loc[102, "name"])
print(by_id["score"].reindex([101, 999]))
print(by_id.reset_index().columns.tolist())
print(by_id.index.is_unique, df.index)

    name  score
id             
101  Ana     88
102   Bo     92
103   Cy     75
Bo
id
101    88.0
999     NaN
Name: score, dtype: float64
['id', 'name', 'score']
True RangeIndex(start=0, stop=3, step=1)


## 17.3 Reading and Writing CSV, Excel and JSON

In [ ]:
import io
import pandas as pd

text = """order_id,customer,amount,placed
1001,ana,25.5,2025-03-01
1002,bo,,2025-03-02
1003,ana,12.0,2025-03-04
"""
df = pd.read_csv(io.StringIO(text))
print(df)
print(df.dtypes)

   order_id customer  amount      placed
0      1001      ana    25.5  2025-03-01
1      1002       bo     NaN  2025-03-02
2      1003      ana    12.0  2025-03-04
order_id      int64
customer        str
amount      float64
placed          str
dtype: object


In [ ]:
import io
import pandas as pd

text = """zip;customer;amount;placed
01234;ana;1,5;2025-03-01
02345;bo;n/a;2025-03-02
"""
df = pd.read_csv(
    io.StringIO(text),
    sep=";",
    dtype={"zip": "str"},
    decimal=",",
    na_values=["n/a"],
    parse_dates=["placed"],
)
print(df)
print(df.dtypes)

     zip customer  amount     placed
0  01234      ana     1.5 2025-03-01
1  02345       bo     NaN 2025-03-02
zip                    str
customer               str
amount             float64
placed      datetime64[us]
dtype: object


In [ ]:
import json
import tempfile
from pathlib import Path
import pandas as pd

df = pd.DataFrame({"id": [1, 2], "region": ["north", "south"], "sales": [10.5, 20.0]})
folder = Path(tempfile.mkdtemp())

df.to_csv(folder / "a.csv", index=False)
print((folder / "a.csv").read_text())

df.to_excel(folder / "a.xlsx", sheet_name="Q1", index=False)
print(pd.read_excel(folder / "a.xlsx", sheet_name="Q1"))

payload = [
    {"id": 1, "user": {"name": "Ana", "city": "Oslo"}, "items": [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 1}]},
    {"id": 2, "user": {"name": "Bo", "city": "Rome"}, "items": [{"sku": "A", "qty": 5}]},
]
print(pd.json_normalize(payload, sep="_").drop(columns="items"))
flat = pd.json_normalize(payload, record_path="items", meta=["id", ["user", "name"]])
print(flat)
print(df.to_json(orient="records"))

id,region,sales
1,north,10.5
2,south,20.0

   id region  sales
0   1  north   10.5
1   2  south   20.0
   id user_name user_city
0   1       Ana      Oslo
1   2        Bo      Rome
  sku  qty id user.name
0   A    2  1       Ana
1   B    1  1       Ana
2   A    5  2        Bo
[{"id":1,"region":"north","sales":10.5},{"id":2,"region":"south","sales":20.0}]


## 17.4 Inspecting a DataFrame

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 12
df = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "city": rng.choice(["Oslo", "Rome", "Lima"], n),
    "amount": rng.normal(50, 20, n).round(2),
})
df.loc[3, "amount"] = np.nan
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)

print(df.shape)
df.info()
print(df.describe().round(2))
print(df["city"].value_counts())
print(df.isna().sum().to_dict(), df.duplicated().sum())

(13, 3)
<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  13 non-null     int64  
 1   city      13 non-null     str    
 2   amount    12 non-null     float64
dtypes: float64(1), int64(1), str(1)
memory usage: 444.0 bytes
       order_id  amount
count     13.00   12.00
mean       6.08   52.99
std        3.77   12.84
min        1.00   30.82
25%        3.00   48.16
50%        6.00   52.56
75%        9.00   60.90
max       12.00   72.54
city
Oslo    5
Lima    4
Rome    4
Name: count, dtype: int64
{'order_id': 0, 'city': 0, 'amount': 1} 1


## 17.5 Selecting: loc, iloc, Boolean Masks and query

In [ ]:
import pandas as pd

df = pd.DataFrame(
    {"city": ["Oslo", "Rome", "Lima", "Oslo"], "sales": [10, 7, 3, 12]},
    index=[10, 11, 12, 13],
)
print(df["city"].tolist())
print(df.loc[11, "sales"])
print(df.iloc[1, 1])
print(df.loc[10:12, "city"].tolist())
print(df.iloc[0:2].index.tolist())

['Oslo', 'Rome', 'Lima', 'Oslo']
7
7
['Oslo', 'Rome', 'Lima']
[10, 11]


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "city": ["Oslo", "Rome", "Lima", "Oslo", "Rome"],
    "sales": [10, 7, 3, 12, 9],
    "paid": [True, False, True, True, False],
})
mask = (df["sales"] > 5) & df["paid"]
print(mask.tolist())
print(df[mask])
print(df[df["city"].isin(["Rome", "Lima"]) | (df["sales"] > 11)])
print(df[~df["paid"]]["sales"].sum())
cut = 8
print(df.query("sales > @cut and city != 'Rome'"))

[True, False, False, True, False]
   city  sales  paid
0  Oslo     10  True
3  Oslo     12  True
   city  sales   paid
1  Rome      7  False
2  Lima      3   True
3  Oslo     12   True
4  Rome      9  False
16
   city  sales  paid
0  Oslo     10  True
3  Oslo     12  True


In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({"x": [-5, 2, -1, 8], "grp": ["a", "b", "a", "b"]})
df.loc[df["x"] < 0, "x"] = 0
print(df["x"].tolist())

df["sign"] = np.where(df["x"] > 0, "pos", "zero")
df["capped"] = df["x"].where(df["x"] < 5, 5)
df["label"] = np.select([df["x"] == 0, df["x"] < 5], ["none", "low"], default="high")
print(df)

[0, 2, 0, 8]
   x grp  sign  capped label
0  0   a  zero       0  none
1  2   b   pos       2   low
2  0   a  zero       0  none
3  8   b   pos       5  high


## 17.6 Assigning Columns and Choosing dtypes

In [ ]:
import pandas as pd

df = pd.DataFrame({"item": ["pen", "book", "lamp"], "price": [2.0, 12.5, 30.0], "qty": [10, 2, 1]})
df["total"] = df["price"] * df["qty"]
print(df)

priced = df.assign(vat=lambda d: (d["total"] * 0.2).round(2), big=lambda d: d["total"] > 20)
print(priced[["item", "vat", "big"]])
print(list(df.columns))

   item  price  qty  total
0   pen    2.0   10   20.0
1  book   12.5    2   25.0
2  lamp   30.0    1   30.0
   item  vat    big
0   pen  4.0  False
1  book  5.0   True
2  lamp  6.0   True
['item', 'price', 'qty', 'total']


In [ ]:
import pandas as pd

raw = pd.Series(["12", "7", "oops", "30", None])
nums = pd.to_numeric(raw, errors="coerce")
print(nums.tolist())

ints = nums.astype("Int64")
print(ints.tolist(), ints.dtype)

size = pd.Series(["M", "S", "L", "M", "S"])
cat = pd.Series(pd.Categorical(size, categories=["S", "M", "L"], ordered=True))
print(cat.dtype, (cat >= "M").tolist())
print(cat.sort_values().tolist(), cat.cat.codes.tolist())

dates = pd.to_datetime(pd.Series(["03/04/2025", "15/04/2025"]), format="%d/%m/%Y")
print(dates.dt.month.tolist(), dates.dtype)

[12.0, 7.0, nan, 30.0, nan]
[12, 7, <NA>, 30, <NA>] Int64
category [True, False, True, True, False]
['S', 'S', 'M', 'M', 'L'] [1, 0, 2, 1, 0]
[4, 4] datetime64[us]


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 200_000
city = pd.Series(rng.choice(["Oslo", "Rome", "Lima", "Kyiv"], n))
qty = pd.Series(rng.integers(0, 100, n))

print(round(city.memory_usage(deep=True) / 1e6, 2), "MB as str")
print(round(city.astype("category").memory_usage(deep=True) / 1e6, 2), "MB as category")
print(round(qty.memory_usage(deep=True) / 1e6, 2), "MB int64")
print(round(qty.astype("int8").memory_usage(deep=True) / 1e6, 2), "MB int8")
print(qty.astype("int8").max(), np.iinfo("int8").max)

10.6 MB as str
0.2 MB as category
1.6 MB int64
0.2 MB int8
99 127


## 17.7 Missing Data and Duplicates

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({"id": [1, 2, 3, 4], "age": [25, np.nan, 40, np.nan], "city": ["Oslo", "Rome", None, "Lima"]})
print(df.isna().sum().to_dict())
print(df.dropna().shape)
print(df.dropna(subset=["city"]).shape)
filled = df.assign(age=df["age"].fillna(df["age"].median()), city=df["city"].fillna("unknown"))
print(filled)

{'id': 0, 'age': 2, 'city': 1}
(1, 3)
(3, 3)
   id   age     city
0   1  25.0     Oslo
1   2  32.5     Rome
2   3  40.0  unknown
3   4  32.5     Lima


In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "dept": ["a", "a", "b", "b", "b"],
    "pay": [100.0, np.nan, 200.0, -999.0, np.nan],
})
df["pay"] = df["pay"].replace(-999, np.nan)
df["pay_filled"] = df["pay"].fillna(df.groupby("dept")["pay"].transform("median"))
print(df)

ts = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan])
print(ts.ffill().tolist())
print(ts.interpolate().tolist())

  dept    pay  pay_filled
0    a  100.0       100.0
1    a    NaN       100.0
2    b  200.0       200.0
3    b    NaN       200.0
4    b    NaN       200.0
[1.0, 1.0, 1.0, 4.0, 4.0]
[1.0, 2.0, 3.0, 4.0, 4.0]


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "id": [1, 1, 2, 3, 3],
    "ts": ["2025-01-01", "2025-01-05", "2025-01-02", "2025-01-03", "2025-01-03"],
    "val": [10, 12, 7, 5, 5],
})
print(df.duplicated().tolist())
print(df.duplicated("id", keep=False).tolist())
latest = df.sort_values("ts").drop_duplicates("id", keep="last")
print(latest.sort_values("id"))
print(df.drop_duplicates().shape[0])

[False, False, False, False, True]
[True, True, False, True, True]
   id          ts  val
1   1  2025-01-05   12
2   2  2025-01-02    7
4   3  2025-01-03    5
4


## 17.8 String and Datetime Accessors, and apply vs Vectorised

In [ ]:
import pandas as pd

names = pd.Series(["  Ana Silva ", "BO chen", "cy O'Neil"])
clean = names.str.strip().str.title()
print(clean.tolist())
parts = clean.str.split(" ", n=1, expand=True)
parts.columns = ["first", "last"]
print(parts)
print(clean.str.contains("an", case=False).tolist())
print(clean.str.len().tolist())

['Ana Silva', 'Bo Chen', "Cy O'Neil"]
  first    last
0   Ana   Silva
1    Bo    Chen
2    Cy  O'Neil
[True, False, False]
[9, 7, 9]


In [ ]:
import pandas as pd

log = pd.DataFrame({
    "line": ["ERR 2025-03-07 14:05 code=502", "OK 2025-03-08 09:30 code=200", "ERR 2025-03-09 23:59 code=404"],
})
ex = log["line"].str.extract(r"^(?P<level>\w+) (?P<when>[\d-]+ [\d:]+) code=(?P<code>\d+)$")
ex["when"] = pd.to_datetime(ex["when"])
ex["code"] = ex["code"].astype(int)
print(ex)
print(ex["when"].dt.day_name().tolist())
ex["day"] = ex["when"].dt.normalize()
ex["hour"] = ex["when"].dt.hour
print(ex[["day", "hour"]])
gap = ex["when"].diff()
print(gap.dt.total_seconds().div(3600).round(1).tolist())

  level                when  code
0   ERR 2025-03-07 14:05:00   502
1    OK 2025-03-08 09:30:00   200
2   ERR 2025-03-09 23:59:00   404
['Friday', 'Saturday', 'Sunday']
         day  hour
0 2025-03-07    14
1 2025-03-08     9
2 2025-03-09    23
[nan, 19.4, 38.5]


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
df = pd.DataFrame({"price": rng.uniform(1, 10, 5), "qty": rng.integers(1, 5, 5)})

a = df["price"] * df["qty"]
b = df.apply(lambda r: r["price"] * r["qty"], axis=1)
c = pd.Series([p * q for p, q in zip(df["price"], df["qty"])])
d = pd.Series([r.price * r.qty for r in df.itertuples()])
print(np.allclose(a, b), np.allclose(a, c), np.allclose(a, d))
print(a.round(2).tolist())

tier = pd.Series([1, 2, 3, 1, 4]).map({1: "single", 2: "pair"}).fillna("bulk")
print(tier.tolist())

True True True
[20.2, 13.71, 4.11, 3.45, 33.28]
['single', 'pair', 'bulk', 'single', 'bulk']


## 17.9 groupby: agg, transform, filter and Named Aggregation

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "city": ["Oslo", "Rome", "Oslo", "Lima", "Rome", "Oslo"],
    "sales": [10, 7, 3, 5, 9, 12],
})
print(df.groupby("city")["sales"].sum())
print(df.groupby("city")["sales"].agg(["count", "mean", "max"]))
print(df.groupby("city", as_index=False)["sales"].sum())

city
Lima     5
Oslo    25
Rome    16
Name: sales, dtype: int64
      count      mean  max
city                      
Lima      1  5.000000    5
Oslo      3  8.333333   12
Rome      2  8.000000    9
   city  sales
0  Lima      5
1  Oslo     25
2  Rome     16


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "city": ["Oslo", "Rome", "Oslo", "Lima", "Rome", "Oslo"],
    "channel": ["web", "web", "shop", "web", "shop", "web"],
    "sales": [10, 7, 3, 5, 9, 12],
})
summary = df.groupby("city").agg(
    orders=("sales", "size"),
    total=("sales", "sum"),
    best=("sales", "max"),
    channels=("channel", "nunique"),
)
print(summary)

df["share_of_city"] = (df["sales"] / df.groupby("city")["sales"].transform("sum")).round(2)
df["vs_city_mean"] = df["sales"] - df.groupby("city")["sales"].transform("mean")
print(df)

big = df.groupby("city").filter(lambda d: len(d) >= 2)
print(sorted(big["city"].unique()))

      orders  total  best  channels
city                               
Lima       1      5     5         1
Oslo       3     25    12         2
Rome       2     16     9         2
   city channel  sales  share_of_city  vs_city_mean
0  Oslo     web     10           0.40      1.666667
1  Rome     web      7           0.44     -1.000000
2  Oslo    shop      3           0.12     -5.333333
3  Lima     web      5           1.00      0.000000
4  Rome    shop      9           0.56      1.000000
5  Oslo     web     12           0.48      3.666667
['Oslo', 'Rome']


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "dept": ["a", "a", "a", "b", "b", "b"],
    "emp": ["u1", "u2", "u3", "u4", "u5", "u6"],
    "pay": [50, 80, 65, 90, 40, 70],
})
top2 = df.sort_values("pay", ascending=False).groupby("dept").head(2)
print(top2.sort_values(["dept", "pay"], ascending=[True, False]))

df["rank"] = df.groupby("dept")["pay"].rank(ascending=False, method="first").astype(int)
print(df[df["rank"] == 1][["dept", "emp"]])

spread = df.groupby("dept")["pay"].agg(lambda s: s.max() - s.min())
print(spread.to_dict())
print(df.groupby("dept")["pay"].agg(["min", "max"]).assign(spread=lambda d: d["max"] - d["min"])["spread"].to_dict())

  dept emp  pay
1    a  u2   80
2    a  u3   65
3    b  u4   90
5    b  u6   70
  dept emp
1    a  u2
3    b  u4
{'a': 30, 'b': 50}
{'a': 30, 'b': 50}


## 17.10 merge, join and concat, with Join Validation

In [ ]:
import pandas as pd

customers = pd.DataFrame({"cid": [1, 2, 3], "name": ["Ana", "Bo", "Cy"]})
orders = pd.DataFrame({"cid": [2, 3, 3, 4], "amount": [20, 30, 5, 99]})

for how in ["inner", "left", "outer"]:
    out = customers.merge(orders, on="cid", how=how)
    print(how, out.shape)
print(customers.merge(orders, on="cid", how="left"))

inner (3, 3)
left (4, 3)
outer (5, 3)
   cid name  amount
0    1  Ana     NaN
1    2   Bo    20.0
2    3   Cy    30.0
3    3   Cy     5.0


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import pandas as pd

users = pd.DataFrame({"user_id": [1, 2, 3], "plan": ["free", "pro", "pro"]})
events = pd.DataFrame({"uid": [1, 1, 2, 9], "event": ["a", "b", "c", "d"]})

m = events.merge(users, left_on="uid", right_on="user_id", how="left", validate="m:1", indicator=True)
print(m)
print(m["_merge"].value_counts().to_dict())

dup_users = pd.concat([users, users.iloc[[0]]], ignore_index=True)
try:
    events.merge(dup_users, left_on="uid", right_on="user_id", validate="m:1")
except Exception as e:
    print(type(e).__name__, "-", str(e)[:60])

   uid event  user_id  plan     _merge
0    1     a      1.0  free       both
1    1     b      1.0  free       both
2    2     c      2.0   pro       both
3    9     d      NaN   NaN  left_only
{'both': 3, 'left_only': 1, 'right_only': 0}
MergeError - Merge keys are not unique in right dataset; not a many-to-on


In [ ]:
import pandas as pd

jan = pd.DataFrame({"id": [1, 2], "v": [10, 20]})
feb = pd.DataFrame({"id": [2, 3], "v": [25, 30]})
stacked = pd.concat([jan, feb], keys=["jan", "feb"])
print(stacked)
print(pd.concat([jan, feb], ignore_index=True).shape)

trades = pd.DataFrame({"ts": pd.to_datetime(["2025-01-01 10:00:05", "2025-01-01 10:00:20"]), "qty": [5, 7]})
quotes = pd.DataFrame({"ts": pd.to_datetime(["2025-01-01 10:00:00", "2025-01-01 10:00:10"]), "px": [100.0, 101.5]})
print(pd.merge_asof(trades, quotes, on="ts"))

print(jan[~jan["id"].isin(feb["id"])]["id"].tolist())

       id   v
jan 0   1  10
    1   2  20
feb 0   2  25
    1   3  30
(4, 2)
                   ts  qty     px
0 2025-01-01 10:00:05    5  100.0
1 2025-01-01 10:00:20    7  101.5
[1]


## 17.11 Reshaping: pivot, pivot_table, melt, stack and crosstab

In [ ]:
import pandas as pd

wide = pd.DataFrame({"id": ["a", "b"], "Q1": [5, 3], "Q2": [7, 4]})
long = wide.melt(id_vars="id", var_name="qtr", value_name="val")
print(long)
back = long.pivot(index="id", columns="qtr", values="val").reset_index()
back.columns.name = None
print(back)
print(back.equals(wide))

  id qtr  val
0  a  Q1    5
1  b  Q1    3
2  a  Q2    7
3  b  Q2    4
  id  Q1  Q2
0  a   5   7
1  b   3   4
True


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "city": ["Oslo", "Oslo", "Rome", "Rome", "Oslo"],
    "year": [2024, 2025, 2024, 2025, 2025],
    "sales": [10, 12, 7, 9, 3],
    "paid": ["y", "n", "y", "y", "y"],
})
pt = df.pivot_table(index="city", columns="year", values="sales", aggfunc="sum", fill_value=0, margins=True)
print(pt)
print(pd.crosstab(df["city"], df["paid"]))
print(pd.crosstab(df["city"], df["paid"], normalize="index").round(2))
s = df.groupby(["city", "year"])["sales"].sum()
print(s.unstack())
print(s.unstack().stack().equals(s))

year  2024  2025  All
city                 
Oslo    10    15   25
Rome     7     9   16
All     17    24   41
paid  n  y
city      
Oslo  1  2
Rome  0  2
paid     n     y
city            
Oslo  0.33  0.67
Rome  0.00  1.00
year  2024  2025
city            
Oslo    10    15
Rome     7     9
True


In [ ]:
import pandas as pd

posts = pd.DataFrame({"id": [1, 2, 3], "tags": ["py|sql", "py", "sql|dbt|py"]})
posts["tag"] = posts["tags"].str.split("|")
exploded = posts.explode("tag")[["id", "tag"]]
print(exploded.groupby("tag").size().sort_values(ascending=False).to_dict())
onehot = posts["tags"].str.get_dummies("|")
print(onehot)
print(pd.get_dummies(pd.Series(["Oslo", "Rome", "Oslo"]), dtype=int))

{'py': 3, 'sql': 2, 'dbt': 1}
   dbt  py  sql
0    0   1    1
1    0   1    0
2    1   1    1
   Oslo  Rome
0     1     0
1     0     1
2     1     0


## 17.12 Window Functions: rolling, expanding and resample

In [ ]:
import pandas as pd

s = pd.Series([10, 12, 9, 15, 20, 18],
              index=pd.date_range("2025-03-01", periods=6, freq="D"), name="visits")
print(s.rolling(3).mean().round(2).tolist())
print(s.rolling(3, min_periods=1).mean().round(2).tolist())
print(s.expanding().max().tolist())
print(s.shift(1).tolist())
print(s.pct_change().round(3).tolist())
print((s - s.shift(7)).isna().all())

[nan, nan, 10.33, 12.0, 14.67, 17.67]
[10.0, 11.0, 10.33, 12.0, 14.67, 17.67]
[10.0, 12.0, 12.0, 15.0, 20.0, 20.0]
[nan, 10.0, 12.0, 9.0, 15.0, 20.0]
[nan, 0.2, -0.25, 0.667, 0.333, -0.1]
True


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(3)
ts = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 90 * 24 * 60, 400), unit="min")
ev = pd.DataFrame({"ts": ts, "amount": rng.integers(5, 100, 400)}).sort_values("ts").set_index("ts")

daily = ev["amount"].resample("D").sum()
print(len(daily), daily.index.min().date(), daily.index.max().date())
monthly = ev["amount"].resample("ME").agg(["count", "sum"])
print(monthly)
print(ev.loc["2025-02-10":"2025-02-11", "amount"].sum())
roll = daily.rolling("7D").sum()
print(roll.iloc[[0, 6, 30]].tolist())
print(ev.groupby(ev.index.dayofweek)["amount"].sum().idxmax())

90 2025-01-01 2025-03-31
            count   sum
ts                     
2025-01-31    140  7064
2025-02-28    122  6418
2025-03-31    138  7147
370
[254.0, 1465.0, 1652.0]
2


## 17.13 MultiIndex, cut/qcut, Method Chaining and pipe

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "city": ["Oslo", "Oslo", "Rome", "Rome"],
    "year": [2024, 2025, 2024, 2025],
    "sales": [10, 15, 7, 9],
}).set_index(["city", "year"]).sort_index()
print(df)
print(df.loc[("Rome", 2025), "sales"])
print(df.xs(2025, level="year"))
print(df.groupby(level="city")["sales"].sum().to_dict())
print(df.reset_index(level="year").columns.tolist())

           sales
city year       
Oslo 2024     10
     2025     15
Rome 2024      7
     2025      9
9
      sales
city       
Oslo     15
Rome      9
{'Oslo': 25, 'Rome': 16}
['year', 'sales']


In [ ]:
import pandas as pd

age = pd.Series([4, 17, 18, 35, 64, 65, 90])
groups = pd.cut(age, bins=[0, 17, 64, 120], labels=["child", "adult", "senior"])
print(groups.tolist())
print(pd.cut(age, bins=[0, 18, 65, 120], right=False).astype(str).tolist()[:3])

spend = pd.Series([5, 8, 12, 20, 33, 50, 80, 400])
q = pd.qcut(spend, 4, labels=["Q1", "Q2", "Q3", "Q4"])
print(q.tolist())
print(q.value_counts().sort_index().to_dict())
print(pd.qcut(spend, 4).cat.categories[0])

['child', 'child', 'adult', 'adult', 'adult', 'senior', 'senior']
['[0, 18)', '[0, 18)', '[18, 65)']
['Q1', 'Q1', 'Q2', 'Q2', 'Q3', 'Q3', 'Q4', 'Q4']
{'Q1': 2, 'Q2': 2, 'Q3': 2, 'Q4': 2}
(4.999, 11.0]


In [ ]:
import pandas as pd

raw = pd.DataFrame({
    "Order ID": [1, 2, 3, 4, 5],
    "Customer": [" ana", "BO ", "ana", "Cy", None],
    "Amount": ["10.5", "7", "oops", "30", "12"],
    "Date": ["2025-03-01", "2025-03-02", "2025-03-02", "2025-03-05", "2025-03-06"],
})

def drop_small(d, min_amount):
    return d[d["amount"] >= min_amount]

clean = (
    raw
    .rename(columns=lambda c: c.strip().lower().replace(" ", "_"))
    .assign(
        customer=lambda d: d["customer"].str.strip().str.lower(),
        amount=lambda d: pd.to_numeric(d["amount"], errors="coerce"),
        date=lambda d: pd.to_datetime(d["date"]),
    )
    .dropna(subset=["customer", "amount"])
    .pipe(drop_small, min_amount=8)
    .sort_values("date")
    .reset_index(drop=True)
)
print(clean)
print(clean.dtypes.astype(str).to_dict())

   order_id customer  amount       date
0         1      ana    10.5 2025-03-01
1         4       cy    30.0 2025-03-05
{'order_id': 'int64', 'customer': 'str', 'amount': 'float64', 'date': 'datetime64[us]'}


## 17.14 Memory, Speed and pandas 3.0 Specifics

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 100_000
df = pd.DataFrame({
    "user_id": rng.integers(0, 5000, n),
    "country": rng.choice(["NO", "IT", "PE", "UA", "US"], n),
    "qty": rng.integers(1, 20, n),
    "price": rng.uniform(1, 100, n).round(2),
})
before = df.memory_usage(deep=True).sum()
slim = df.assign(
    user_id=pd.to_numeric(df["user_id"], downcast="unsigned"),
    country=df["country"].astype("category"),
    qty=pd.to_numeric(df["qty"], downcast="unsigned"),
    price=df["price"].astype("float32"),
)
after = slim.memory_usage(deep=True).sum()
print(slim.dtypes.astype(str).to_dict())
print(f"{before/1e6:.1f} MB -> {after/1e6:.1f} MB")
print(round(after / before, 2))

{'user_id': 'uint16', 'country': 'category', 'qty': 'uint8', 'price': 'float32'}
7.5 MB -> 0.8 MB
0.11


In [ ]:
import pandas as pd

df = pd.DataFrame({"a": [1, 2, 3], "b": ["x", "y", "z"]})
view = df["a"]
view.iloc[0] = 100
print(df["a"].tolist(), view.tolist())

sub = df[df["a"] > 1]
sub.loc[sub.index[0], "a"] = -1
print(df["a"].tolist())

df.loc[df["a"] > 2, "a"] = 0
print(df["a"].tolist())

s = df["b"]
print(s.dtype, s.str.upper().dtype)
print(pd.Series(["a", None]).isna().tolist(), pd.Series(["a", None]).dtype)
print(pd.Series(["a", 1]).dtype)

[1, 2, 3] [100, 2, 3]
[1, 2, 3]
[1, 2, 0]
str str
[False, True] str
object


## 17.15 Recipes: "How do I ...?"

Each recipe is a complete, runnable answer. Copy the pattern, then change the column names.

In [ ]:
import io
import numpy as np
import pandas as pd

text = """Order ID,Customer Name,Country,Amount,Order Date,Status
1001, Ana Silva ,no,"1,250.50",2025-03-01,paid
1002,bo chen,NO,80,2025-03-02,PAID
1002,bo chen,NO,80,2025-03-02,PAID
1003,Cy ONeil,it,-5,2025-03-03,refunded
1004,,IT,,2025-03-04,paid
1005,Di Park,pe,300,not a date,Paid
"""

def clean_orders(raw: pd.DataFrame) -> pd.DataFrame:
    return (
        raw
        .rename(columns=lambda c: c.strip().lower().replace(" ", "_"))
        .assign(
            customer_name=lambda d: d["customer_name"].str.strip().str.title(),
            country=lambda d: d["country"].str.upper(),
            status=lambda d: d["status"].str.lower(),
            amount=lambda d: pd.to_numeric(d["amount"].astype("str").str.replace(",", ""), errors="coerce"),
            order_date=lambda d: pd.to_datetime(d["order_date"], errors="coerce"),
        )
        .drop_duplicates()
        .assign(amount_ok=lambda d: d["amount"].ge(0))
    )

raw = pd.read_csv(io.StringIO(text), dtype={"Amount": "str"})
out = clean_orders(raw)
print(len(raw), "->", len(out))
print(out)
print(out.isna().sum().to_dict())
print(out.loc[out["amount_ok"] & out["order_date"].notna(), "amount"].sum())

6 -> 5
   order_id customer_name country  amount order_date    status  amount_ok
0      1001     Ana Silva      NO  1250.5 2025-03-01      paid       True
1      1002       Bo Chen      NO    80.0 2025-03-02      paid       True
3      1003      Cy Oneil      IT    -5.0 2025-03-03  refunded      False
4      1004           NaN      IT     NaN 2025-03-04      paid      False
5      1005       Di Park      PE   300.0        NaT      paid       True
{'order_id': 0, 'customer_name': 1, 'country': 0, 'amount': 1, 'order_date': 1, 'status': 0, 'amount_ok': 0}
1330.5


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(5)
rows = []
for cust in range(300):
    start = rng.integers(0, 4)
    for m in range(start, 6):
        if m == start or rng.random() < 0.5 ** (m - start) * 0.8 + 0.1:
            rows.append((cust, pd.Timestamp("2025-01-01") + pd.DateOffset(months=int(m))))
orders = pd.DataFrame(rows, columns=["customer", "order_month"])

first = orders.groupby("customer")["order_month"].transform("min")
orders["cohort"] = first.dt.to_period("M")
orders["age"] = (orders["order_month"].dt.year - first.dt.year) * 12 + (orders["order_month"].dt.month - first.dt.month)

counts = orders.pivot_table(index="cohort", columns="age", values="customer", aggfunc="nunique")
retention = counts.div(counts[0], axis=0).round(2)
print(retention)
print(counts[0].tolist())

age        0     1     2     3     4     5
cohort                                    
2025-01  1.0  0.53  0.22  0.17  0.18  0.09
2025-02  1.0  0.52  0.36  0.19  0.04   NaN
2025-03  1.0  0.58  0.36  0.19   NaN   NaN
2025-04  1.0  0.53  0.27   NaN   NaN   NaN
[90.0, 75.0, 73.0, 62.0]


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
n = 400
orders = pd.DataFrame({
    "customer": rng.integers(0, 60, n),
    "date": pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "amount": rng.gamma(2.0, 30.0, n).round(2),
})
today = orders["date"].max() + pd.Timedelta(days=1)

rfm = orders.groupby("customer").agg(
    recency=("date", lambda s: (today - s.max()).days),
    frequency=("date", "size"),
    monetary=("amount", "sum"),
)
rfm["r"] = pd.qcut(rfm["recency"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["f"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["m"] = pd.qcut(rfm["monetary"], 4, labels=[1, 2, 3, 4]).astype(int)
rfm["score"] = rfm["r"] + rfm["f"] + rfm["m"]
print(rfm.sort_values("score", ascending=False).head(4).round(1))
print(rfm["score"].describe().round(2)[["min", "max"]].tolist())
print(rfm["r"].value_counts().sort_index().tolist())

          recency  frequency  monetary  r  f  m  score
customer                                              
17             10         10     776.8  4  4  4     12
58              1          9     650.8  4  4  4     12
4               7         11     571.9  4  4  4     12
54             10          8     663.1  4  4  4     12
[3.0, 12.0]
[14, 16, 13, 17]


In [ ]:
import pandas as pd

sales = pd.DataFrame({
    "region": ["n", "n", "n", "s", "s", "s", "s"],
    "product": ["a", "b", "c", "a", "b", "c", "d"],
    "revenue": [100, 250, 80, 40, 40, 300, 90],
})
top2 = (sales.sort_values(["region", "revenue"], ascending=[True, False])
             .groupby("region").head(2))
print(top2)
ranked = sales.assign(rk=sales.groupby("region")["revenue"].rank(method="dense", ascending=False))
print(ranked[ranked["rk"] <= 2][["region", "product", "rk"]])
best = sales.loc[sales.groupby("region")["revenue"].idxmax()]
print(best[["region", "product"]].to_dict("records"))

  region product  revenue
1      n       b      250
0      n       a      100
5      s       c      300
6      s       d       90
  region product   rk
0      n       a  2.0
1      n       b  1.0
5      s       c  1.0
6      s       d  2.0
[{'region': 'n', 'product': 'b'}, {'region': 's', 'product': 'c'}]


In [ ]:
import pandas as pd

ev = pd.DataFrame({
    "user": ["u1", "u1", "u1", "u1", "u2", "u2"],
    "ts": pd.to_datetime(["2025-03-01 09:00", "2025-03-01 09:10", "2025-03-01 10:00",
                          "2025-03-01 10:05", "2025-03-01 09:00", "2025-03-01 09:29"]),
}).sort_values(["user", "ts"])

gap = ev.groupby("user")["ts"].diff()
new_session = gap.isna() | (gap > pd.Timedelta(minutes=30))
ev["session"] = new_session.groupby(ev["user"]).cumsum()
print(ev)
summary = ev.groupby(["user", "session"]).agg(start=("ts", "min"), end=("ts", "max"), events=("ts", "size"))
summary["minutes"] = (summary["end"] - summary["start"]).dt.total_seconds() / 60
print(summary)

  user                  ts  session
0   u1 2025-03-01 09:00:00        1
1   u1 2025-03-01 09:10:00        1
2   u1 2025-03-01 10:00:00        2
3   u1 2025-03-01 10:05:00        2
4   u2 2025-03-01 09:00:00        1
5   u2 2025-03-01 09:29:00        1
                           start                 end  events  minutes
user session                                                         
u1   1       2025-03-01 09:00:00 2025-03-01 09:10:00       2     10.0
     2       2025-03-01 10:00:00 2025-03-01 10:05:00       2      5.0
u2   1       2025-03-01 09:00:00 2025-03-01 09:29:00       2     29.0


In [ ]:
import pandas as pd

ledger = pd.DataFrame({"invoice": ["A1", "A2", "A3", "A4"], "amount": [100.0, 250.0, 75.0, 60.0]})
bank = pd.DataFrame({"invoice": ["A2", "A3", "A5"], "amount": [250.0, 70.0, 40.0]})

m = ledger.merge(bank, on="invoice", how="outer", suffixes=("_ledger", "_bank"), indicator=True, validate="1:1")
only_ledger = m.loc[m["_merge"] == "left_only", "invoice"].tolist()
only_bank = m.loc[m["_merge"] == "right_only", "invoice"].tolist()
both = m[m["_merge"] == "both"].assign(diff=lambda d: d["amount_ledger"] - d["amount_bank"])
mismatch = both[both["diff"].abs() > 0.005]
print(only_ledger, only_bank)
print(mismatch[["invoice", "amount_ledger", "amount_bank", "diff"]])
print(m["_merge"].value_counts().to_dict())

['A1', 'A4'] ['A5']
  invoice  amount_ledger  amount_bank  diff
2      A3           75.0         70.0   5.0
{'left_only': 2, 'both': 2, 'right_only': 1}


In [ ]:
import sqlite3
import pandas as pd

t = pd.DataFrame({"k": ["a", "a", "b", "b", "b"], "x": [3, 9, 4, 4, 1], "d": [1, 2, 1, 2, 3]})
con = sqlite3.connect(":memory:")
t.to_sql("t", con, index=False)

sql = pd.read_sql("""
    SELECT k, SUM(x) AS s, COUNT(*) AS n
    FROM t GROUP BY k HAVING SUM(x) > 8 ORDER BY s DESC""", con)
pdf = (t.groupby("k", as_index=False).agg(s=("x", "sum"), n=("x", "size"))
        .query("s > 8").sort_values("s", ascending=False).reset_index(drop=True))
print(sql.equals(pdf))
print(pdf)

win = pd.read_sql("SELECT k, d, x, ROW_NUMBER() OVER (PARTITION BY k ORDER BY x DESC, d) AS rn FROM t ORDER BY k, rn", con)
t2 = t.sort_values(["k", "x", "d"], ascending=[True, False, True])
t2["rn"] = t2.groupby("k").cumcount() + 1
print(win.equals(t2[["k", "d", "x", "rn"]].reset_index(drop=True)))

True
   k   s  n
0  a  12  2
1  b   9  3
True


In [ ]:
import pandas as pd

sales = pd.DataFrame({
    "product": ["a", "b", "c", "a", "b", "d"],
    "month": ["jan", "jan", "jan", "feb", "feb", "feb"],
    "units": [100, 50, 20, 130, 40, 10],
})
wide = sales.pivot(index="product", columns="month", values="units").fillna(0)
wide["change"] = wide["feb"] - wide["jan"]
wide["pct"] = ((wide["change"] / wide["jan"]).replace([float("inf"), -float("inf")], float("nan")) * 100).round(1)
print(wide.sort_values("change", ascending=False))

month      feb    jan  change    pct
product                             
a        130.0  100.0    30.0   30.0
d         10.0    0.0    10.0    NaN
b         40.0   50.0   -10.0  -20.0
c          0.0   20.0   -20.0 -100.0


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(1)
s = pd.Series(np.append(rng.normal(100, 10, 200), [250, -40]), name="x")
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
out = s[(s < low) | (s > high)]
print(round(low, 1), round(high, 1))
print(out.round(1).to_dict())
z = (s - s.mean()) / s.std()
print((z.abs() > 3).sum())
clipped = s.clip(low, high)
print(round(clipped.min(), 1), round(clipped.max(), 1))

74.9 123.8
{24: 72.9, 124: 125.5, 183: 128.7, 193: 72.9, 200: 250.0, 201: -40.0}
2
74.9 123.8


In [ ]:
import pandas as pd

s = pd.Series([5, 8, 6], index=pd.to_datetime(["2025-03-03", "2025-03-05", "2025-03-10"]), name="orders")
daily = s.asfreq("D", fill_value=0)
print(daily.tolist())
weekly = daily.resample("W-SUN").sum()
print(weekly.to_dict())
print(weekly.pct_change().round(3).tolist())

[5, 0, 8, 0, 0, 0, 0, 6]
{Timestamp('2025-03-09 00:00:00'): 13, Timestamp('2025-03-16 00:00:00'): 6}
[nan, -0.538]


In [ ]:
import tempfile
from pathlib import Path
import pandas as pd

folder = Path(tempfile.mkdtemp())
for month in ["2025-01", "2025-02", "2025-03"]:
    pd.DataFrame({"id": [1, 2], "v": [10, 20]}).to_csv(folder / f"sales_{month}.csv", index=False)

parts = [pd.read_csv(p).assign(source=p.stem) for p in sorted(folder.glob("sales_*.csv"))]
all_data = pd.concat(parts, ignore_index=True)
print(all_data.shape)
print(all_data["source"].unique().tolist())

(6, 3)
['sales_2025-01', 'sales_2025-02', 'sales_2025-03']


In [ ]:
import pandas as pd

old = pd.DataFrame({"id": [1, 2, 3], "price": [10.0, 20.0, 30.0], "stock": [5, 0, 7]}).set_index("id")
new = pd.DataFrame({"id": [1, 2, 4], "price": [10.0, 22.0, 15.0], "stock": [5, 0, 3]}).set_index("id")

added = new.index.difference(old.index).tolist()
removed = old.index.difference(new.index).tolist()
common = old.index.intersection(new.index)
diff = old.loc[common].compare(new.loc[common])
print(added, removed)
print(diff)

[4] [3]
   price      
    self other
id            
2   20.0  22.0


## 17.16 Performance and Pitfalls

Most pandas disasters are not crashes; they are quiet wrong answers. Each pitfall below has the same layout: a red box describing the trap, then one block that runs the *wrong* code, shows the symptom, and runs the *fix*.

In [ ]:
import pandas as pd

df = pd.DataFrame({"a": [5, -1, 7]})
df[df["a"] < 0]["a"] = 0          # wrong: edits a temporary copy
print("wrong:", df["a"].tolist())
df.loc[df["a"] < 0, "a"] = 0      # fix: one .loc assignment on df itself
print("fixed:", df["a"].tolist())

wrong: [5, -1, 7]
fixed: [5, 0, 7]


In [ ]:
import pandas as pd

df = pd.DataFrame({"a": [1.0, None, 3.0]})
res = df.dropna(inplace=True)      # wrong to assign this
print("wrong:", res)
df = pd.DataFrame({"a": [1.0, None, 3.0]})
df = df.dropna()                    # fix
print("fixed:", len(df))

wrong: None
fixed: 2


In [ ]:
import pandas as pd

orders = pd.DataFrame({"cid": [1, 2], "amount": [100, 50]})
cust = pd.DataFrame({"cid": [1, 1, 2], "tier": ["gold", "gold", "basic"]})
bad = orders.merge(cust, on="cid", how="left")
print("wrong total:", bad["amount"].sum())
try:
    orders.merge(cust, on="cid", how="left", validate="m:1")
except Exception as e:
    print("caught:", type(e).__name__)
good = orders.merge(cust.drop_duplicates("cid"), on="cid", how="left", validate="m:1")
print("fixed total:", good["amount"].sum())

wrong total: 250
caught: MergeError
fixed total: 150


In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({"x": [1.0, np.nan, 3.0]})
print("wrong:", len(df[df["x"] == np.nan]))
print("fixed:", len(df[df["x"].isna()]))

wrong: 0
fixed: 1


In [ ]:
import pandas as pd

s = pd.Series([100, 200, 300])
print("wrong:", s.astype("int8").tolist())
print("fixed:", pd.to_numeric(s, downcast="integer").dtype)

wrong: [100, -56, 44]
fixed: int16


In [ ]:
import pandas as pd

df = pd.DataFrame({"k": ["a", None, "a", "b"], "v": [1, 2, 3, 4]})
print("wrong:", df.groupby("k")["v"].sum().sum(), "of", df["v"].sum())
print("fixed:", df.groupby("k", dropna=False)["v"].sum().sum())

wrong: 8 of 10
fixed: 10


In [ ]:
import pandas as pd

df = pd.DataFrame({"n": [1, 2], "x": [1.5, 2.5]})
for _, row in df.iterrows():
    print("wrong:", type(row["n"]).__name__, end="  ")
print()
for row in df.itertuples():
    print("fixed:", type(row.n).__name__, end="  ")
print()

wrong: float64  wrong: float64  
fixed: int  fixed: int  


In [ ]:
import pandas as pd

s = pd.Series(["01/02/2025"])
print("wrong:", pd.to_datetime(s)[0].date())
print("fixed:", pd.to_datetime(s, format="%d/%m/%Y")[0].date())

wrong: 2025-01-02
fixed: 2025-02-01


In [ ]:
import pandas as pd

df = pd.DataFrame({"a": [1, 2, 3]}, index=[10, 11, 12])
other = pd.Series([7, 8, 9])                 # index 0, 1, 2
df["b"] = other
print("wrong:", df["b"].tolist())
df["b"] = other.to_numpy()
print("fixed:", df["b"].tolist())

wrong: [nan, nan, nan]
fixed: [7, 8, 9]


In [ ]:
import pandas as pd

df = pd.DataFrame({"qty": ["3", "4", "x"]})
try:
    df["qty"].mean()
except TypeError as e:
    print("wrong:", type(e).__name__)
qty = pd.to_numeric(df["qty"], errors="coerce")
print("fixed:", qty.mean(), "with", int(qty.isna().sum()), "bad value")

wrong: TypeError
fixed: 3.5 with 1 bad value


In [ ]:
import numpy as np
import pandas as pd

s = pd.Series([0.1, 0.2])
print("wrong:", s.sum() == 0.3, repr(s.sum()))
print("fixed:", bool(np.isclose(s.sum(), 0.3)), (pd.Series([10, 20]).sum() == 30))

wrong: False np.float64(0.30000000000000004)
fixed: True True


In [ ]:
import pandas as pd

rows = []
for i in range(1000):
    rows.append({"i": i, "sq": i * i})
df = pd.DataFrame(rows)          # fix: one construction at the end
print(df.shape, df["sq"].iloc[-1])

(1000, 2) 998001


In [ ]:
import time
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 300_000
df = pd.DataFrame({"k": rng.integers(0, 1000, n), "x": rng.normal(size=n)})

def timeit(f):
    t = time.perf_counter(); f(); return time.perf_counter() - t

fast = timeit(lambda: df.groupby("k")["x"].agg("sum"))
slow = timeit(lambda: df.groupby("k")["x"].agg(lambda s: s.sum()))
print("lambda slower than built-in:", slow > fast)
a = df.groupby("k")["x"].agg("sum")
b = df.groupby("k")["x"].agg(lambda s: s.sum())
print(np.allclose(a, b))

lambda slower than built-in: True
True


## 17.17 Cross-library Integration

pandas rarely works alone. The arrows below are the handoffs you will use most.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
df = pd.DataFrame({"ad": rng.uniform(0, 10, 200)})
df["sales"] = 3 * df["ad"] + 5 + rng.normal(0, 1, 200)

con = sqlite3.connect(":memory:")
df.to_sql("t", con, index=False)
big = pd.read_sql("SELECT * FROM t WHERE ad > 5", con)

X_tr, X_te, y_tr, y_te = train_test_split(df[["ad"]], df["sales"], test_size=0.25, random_state=0)
model = LinearRegression().fit(X_tr, y_tr)
pred = pd.Series(model.predict(X_te), index=X_te.index, name="pred")
res = pd.concat([X_te, y_te, pred], axis=1)
print(len(big), res.shape)
print(round(float(model.coef_[0]), 2), round(float(model.intercept_), 2))
print(res.head(2).round(2))

import matplotlib.pyplot as plt
ax = res.plot.scatter("ad", "sales", color="#2b4fd6")
print(type(ax).__name__)
print(np.log1p(df["ad"]).head(2).round(3).tolist())

112 (50, 3)
2.95 5.15
       ad  sales   pred
18   3.00  14.56  14.00
170  3.44  14.68  15.32
Axes
[1.997, 1.308]


## 17.18 How to Solve Any pandas Problem

In [ ]:
import pandas as pd

o = pd.DataFrame({
    "cust": ["a", "a", "a", "b", "b", "c"],
    "day": pd.to_datetime(["2025-03-01", "2025-03-05", "2025-03-20", "2025-03-02", "2025-03-03", "2025-03-09"]),
    "amt": [10, 30, 20, 50, 5, 40],
})
regions = pd.DataFrame({"cust": ["a", "b", "d"], "region": ["north", "south", "west"]})

A = o.groupby("cust")["amt"].sum()
B = (o["amt"] / o.groupby("cust")["amt"].transform("sum")).round(2)
C = o.sort_values("day").drop_duplicates("cust", keep="last")[["cust", "amt"]]
D = o.merge(regions, on="cust", how="left", validate="m:1")
E = regions.merge(o, on="cust", how="left", indicator=True).query("_merge == 'left_only'")["cust"].tolist()
F = o.pivot_table(index="cust", columns=o["day"].dt.to_period("M"), values="amt", aggfunc="sum")
G = o.sort_values(["cust", "day"]).assign(days_since=lambda d: d.groupby("cust")["day"].diff().dt.days)
H = pd.cut(o["amt"], [0, 10, 30, 100], labels=["small", "mid", "large"])

print(A.to_dict())
print(B.tolist())
print(C.to_dict("records"))
print(D["region"].tolist())
print(E)
print(F.round(0).to_dict())
print(G["days_since"].tolist())
print(H.tolist())

{'a': 60, 'b': 55, 'c': 40}
[0.17, 0.5, 0.33, 0.91, 0.09, 1.0]
[{'cust': 'b', 'amt': 5}, {'cust': 'c', 'amt': 40}, {'cust': 'a', 'amt': 20}]
['north', 'north', 'north', 'south', 'south', nan]
['d']
{Period('2025-03', 'M'): {'a': 60, 'b': 55, 'c': 40}}
[nan, 4.0, 15.0, nan, 1.0, nan]
['small', 'mid', 'mid', 'large', 'small', 'large']


In [ ]:
import pandas as pd

df = pd.DataFrame({"city": ["Oslo", "oslo ", "Rome"], "x": ["1", "2", "3"]})
steps = (
    df
    .assign(city=lambda d: d["city"].str.strip().str.lower())
    .pipe(lambda d: (print("after clean:", d["city"].tolist()), d)[1])
    .assign(x=lambda d: pd.to_numeric(d["x"]))
    .groupby("city", as_index=False)["x"].sum()
)
print(steps.to_dict("records"))
print(df["x"].dtype, df.shape, df.index.is_unique)

after clean: ['oslo', 'oslo', 'rome']
[{'city': 'oslo', 'x': 3}, {'city': 'rome', 'x': 3}]
str (3, 2) True


## 17.19 Going Deeper: Internals, Memory, Speed and What Interviewers Probe

In [ ]:
import numpy as np
import pandas as pd

codes, uniques = pd.factorize(pd.Series(["b", "a", "b", "c"]))
print(codes.tolist(), list(uniques))

df = pd.DataFrame({"a": [1, 2, 3]})
shallow = df.copy(deep=False)
print(np.shares_memory(df["a"].to_numpy(), shallow["a"].to_numpy()))
shallow.loc[0, "a"] = 99
print(np.shares_memory(df["a"].to_numpy(), shallow["a"].to_numpy()), df["a"].tolist())

idx = pd.Index(["x", "y"])
try:
    idx[0] = "z"
except TypeError as e:
    print("Index is immutable:", type(e).__name__)

[0, 1, 0, 2] ['b', 'a', 'c']
True
False [1, 2, 3]
Index is immutable: TypeError


In [ ]:
import io
import pandas as pd

lines = ["user,amount"] + [f"u{i % 4},{i}" for i in range(1, 21)]
text = "\n".join(lines)

totals = {}
for chunk in pd.read_csv(io.StringIO(text), chunksize=6):
    part = chunk.groupby("user")["amount"].sum()
    for user, value in part.items():
        totals[user] = totals.get(user, 0) + int(value)
print(totals)
print(sum(totals.values()), sum(range(1, 21)))

{'u0': 60, 'u1': 45, 'u2': 50, 'u3': 55}
210 210


## Chapter recap

## Practice set

Graded from easy to expert. Try each one before reading the answer. Outputs shown were produced by running the code.

In [ ]:
import pandas as pd

s = pd.Series([4, 8, 15], index=["a", "b", "c"])
print(s["b"], s.iloc[2])
print(s.loc["a":"b"].tolist())

In [ ]:
import pandas as pd

df = pd.DataFrame({"a": [1, 5, 9], "b": [2, 2, 8]})
try:
    print(df[(df["a"] > 2) and (df["b"] < 5)])
except ValueError as e:
    print("ValueError")
print(df[(df["a"] > 2) & (df["b"] < 5)])

In [ ]:
import pandas as pd

a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20], index=["y", "z"])
print((a + b).tolist())
print(a.add(b, fill_value=0).tolist())

In [ ]:
import pandas as pd

L = pd.DataFrame({"k": [1, 2, 2, 3]})
R = pd.DataFrame({"k": [2, 2, 3, 4]})
print({how: len(L.merge(R, on="k", how=how)) for how in ["inner", "left", "outer"]})

In [ ]:
import pandas as pd

df = pd.DataFrame({"v": [3, -2, 5, -7]})
df["v"][df["v"] < 0] = 0
print(df["v"].tolist())
df.loc[df["v"] < 0, "v"] = 0
print(df["v"].tolist())

In [ ]:
import pandas as pd

s = pd.Series([1, 2, 3, 4, 5])
print(s.rolling(3).sum().tolist())
print(s.shift(1).tolist())
print(s.diff().tolist())
print(s.expanding().mean().tolist())

In [ ]:
import pandas as pd

age = pd.Series([18, 64, 65])
print(pd.cut(age, [0, 18, 64, 120]).astype(str).tolist())
print(pd.cut(age, [0, 18, 64, 120], right=False).astype(str).tolist())

In [ ]:
import pandas as pd

orders = pd.DataFrame({"cid": [1, 2, 3], "amt": [100, 200, 300]})
tiers = pd.DataFrame({"cid": [1, 2, 2, 3], "tier": ["g", "s", "s", "b"]})
bad = orders.merge(tiers, on="cid")
print(bad.groupby("tier")["amt"].sum().to_dict())
good = orders.merge(tiers.drop_duplicates("cid"), on="cid", validate="m:1")
print(good.groupby("tier")["amt"].sum().to_dict())

In [ ]:
import pandas as pd

df = pd.DataFrame({"dept": list("aaabbb"), "emp": ["u1", "u2", "u3", "u4", "u5", "u6"], "pay": [50, 80, 80, 90, 40, 70]})
r = df.sort_values(["dept", "pay", "emp"], ascending=[True, False, True]).groupby("dept").head(2)
print(r[["dept", "emp", "pay"]].to_dict("records"))

In [ ]:
import pandas as pd

ts = pd.to_datetime(["2025-01-01 09:00", "2025-01-01 09:10", "2025-01-01 09:30", "2025-01-01 09:35"])
ev = pd.DataFrame({"user": "u1", "ts": ts})
gap = ev.groupby("user")["ts"].diff()
ev["session"] = (gap.isna() | (gap > pd.Timedelta(minutes=15))).groupby(ev["user"]).cumsum()
print(ev.groupby("session").size().to_dict())

In [ ]:
import pandas as pd

df = pd.DataFrame({"region": ["n", "s", None, "n"], "rev": [10, 20, 30, 40]})
print(df.groupby("region")["rev"].sum().sum(), df["rev"].sum())
print(df.groupby("region", dropna=False)["rev"].sum().sum())

In [ ]:
import pandas as pd

df = pd.DataFrame({"a": [1, 2, 3]})
v = df["a"]
w = df.copy(deep=False)
v.iloc[0] = 10
w.loc[1, "a"] = 20
print(df["a"].tolist())
print(v.tolist(), w["a"].tolist())
s = pd.Series(["x", None, "y"])
print(s.dtype, s.isna().sum(), s.str.upper().tolist())

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch17.html